In [7]:
# Cell 1 - Imports

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [8]:
# Cell 2 - Load WeatherAUS

df = pd.read_csv("../data/weatherAUS.csv")

print(df.shape)
df.head()

(145460, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [9]:


df = df.dropna(subset=["RainTomorrow"])

print(df.shape)
df["RainTomorrow"] = df["RainTomorrow"].map({
    "No": 0,
    "Yes": 1
})

df["RainTomorrow"].value_counts()
df = df.drop(columns=["Date"])

df.head()

(142193, 23)


,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,WNW,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,0
1,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,WSW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,0
2,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,WSW,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,0
3,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,E,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,0
4,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,NW,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,0


In [10]:

x = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

print(x.shape)
print(y.shape)
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", x_train.shape)
print("Testing:", x_test.shape)

(142193, 21)
(142193,)
Training: (113754, 21)
Testing: (28439, 21)


In [11]:
numeric_columns = x_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = x_train.select_dtypes(
    include=["object"]
).columns

print("Numeric columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))

Numeric columns: 16
Categorical columns: 5


C:\Users\kunap\AppData\Local\Temp\ipykernel_9924\186679957.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = x_train.select_dtypes(


In [12]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=800,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)


rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("random_forest", rf_model)
    ]
)

rf_pipeline.fit(
    x_train,
    y_train
)

print("Training complete")

Training complete


In [14]:
rf_predictions = rf_pipeline.predict(x_test)
print("Accuracy:", accuracy_score(y_test, rf_predictions))

print("\nClassification Report:")
print(classification_report(y_test, rf_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))

Accuracy: 0.8495024438271388

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.90      0.90     22064
           1       0.66      0.67      0.66      6375

    accuracy                           0.85     28439
   macro avg       0.78      0.78      0.78     28439
weighted avg       0.85      0.85      0.85     28439


Confusion Matrix:
[[19915  2149]
 [ 2131  4244]]


In [15]:
train_predictions = rf_pipeline.predict(x_train)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_predictions)
)

print(
    "Testing Accuracy:",
    accuracy_score(y_test, rf_predictions)
)

Training Accuracy: 0.9992439826291822
Testing Accuracy: 0.8495024438271388


In [16]:
rf_model_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_pipeline_tuned = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("random_forest", rf_model_tuned)
    ]
)

rf_pipeline_tuned.fit(x_train, y_train)

train_predictions_tuned = rf_pipeline_tuned.predict(x_train)
test_predictions_tuned = rf_pipeline_tuned.predict(x_test)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_predictions_tuned)
)

print(
    "Testing Accuracy:",
    accuracy_score(y_test, test_predictions_tuned)
)

print("\nClassification Report:")
print(classification_report(y_test, test_predictions_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions_tuned))

Training Accuracy: 0.8415176609174183
Testing Accuracy: 0.8084672456837442

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.82      0.87     22064
           1       0.55      0.77      0.64      6375

    accuracy                           0.81     28439
   macro avg       0.74      0.79      0.76     28439
weighted avg       0.84      0.81      0.82     28439


Confusion Matrix:
[[18083  3981]
 [ 1466  4909]]


In [ ]:
rf_model_tuned2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_pipeline_tuned2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("random_forest", rf_model_tuned2)
    ]
)

rf_pipeline_tuned2.fit(x_train, y_train)

train_predictions_tuned2 = rf_pipeline_tuned2.predict(x_train)
test_predictions_tuned2 = rf_pipeline_tuned2.predict(x_test)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_predictions_tuned2)
)

print(
    "Testing Accuracy:",
    accuracy_score(y_test, test_predictions_tuned2)
)

print("\nClassification Report:")
print(classification_report(y_test, test_predictions_tuned2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions_tuned2))